# EX_16: RPS Classification — 경량화 + 정확도 균형
## MobileNetV3Large + Pruning (50%) + QAT INT8

| 항목 | EX_15 (EfficientNetB3) | EX_16 (MobileNetV3Large) |
|------|------------------------|--------------------------|
| 목표 | 정확도 극대화 | **경량화 + 정확도 균형** |
| 백본 | EfficientNetB3 (12M) | **MobileNetV3Large (5.4M)** |
| IMG_SIZE | 300 | **224** (MV3 권장) |
| 증강 | Mixup | **Mixup** (유지) |
| Loss | Label Smoothing 0.1 | **Label Smoothing 0.1 → 0.05** |
| 경량화 기법 | Float32 TFLite | **Pruning 50% + QAT INT8** |
| Fine-tuning | 3단계 | **4단계 (Pruning + QAT 추가)** |
| 예상 크기 | ~48 MB | **~2–3 MB** |
| 목표 정확도 | 0.99+ | **0.95+** |

## Step 0. 패키지 설치
> ⚠️ 설치 후 **런타임 재시작** 필수

In [ ]:
!pip install tensorflow-model-optimization -q

## Step 1. 모듈 로딩

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob, cv2, pathlib, shutil
from sklearn.model_selection import train_test_split
import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

print('NumPy      :', np.__version__)
print('TensorFlow :', tf.__version__)
print('GPU        :', tf.config.list_physical_devices('GPU'))

## Step 2. Google Drive 마운트

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted.')
    colab = True
except:
    print('Local environment.')
    colab = False

## Step 3. 데이터셋 준비
- `IMG_SIZE = 224` : MobileNetV3Large 기본 해상도
- MobileNetV3는 `include_preprocessing=True` 기본 → raw `[0,255]` 그대로 전달
- `test_labels_oh` : Phase 2~4 one-hot, `test_labels` : sparse (평가/시각화용)

In [ ]:
files_path  = '/content/drive/MyDrive/sds/RPS_Dataset'
IMG_SIZE    = 224
NUM_CLASSES = 3
class_names = ['Rock', 'Paper', 'Scissors']

print(f'데이터 로딩 중... (IMG_SIZE={IMG_SIZE})')
all_images, all_labels = [], []

for label in range(NUM_CLASSES):
    files = glob.glob(f'{files_path}/{label}/*.*')
    print(f'  [{label}] {class_names[label]}: {len(files)}개')
    for f in files:
        img = cv2.imread(f, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        all_images.append(img)
        all_labels.append(label)

all_images = np.array(all_images, dtype=np.uint8)
all_labels = np.array(all_labels)

train_data, test_data, train_labels, test_labels = train_test_split(
    all_images, all_labels,
    test_size=0.2, random_state=123, stratify=all_labels
)

print(f'\nTrain: {train_data.shape}, Test: {test_data.shape}')
print(f'클래스 분포 (train): {np.bincount(train_labels)}')
print(f'클래스 분포 (test) : {np.bincount(test_labels)}')

train_data_f = train_data.astype(np.float32)
test_data_f  = test_data.astype(np.float32)

test_labels_oh = np.eye(NUM_CLASSES)[test_labels]
print(f'test_labels shape    : {test_labels.shape}    ← sparse (평가/시각화용)')
print(f'test_labels_oh shape : {test_labels_oh.shape} ← one-hot (Phase2~4 validation용)')

## Step 4. Mixup 증강 + 제너레이터 정의
- **Mixup**: 두 이미지를 랜덤 비율로 섞어 soft label 생성 → 일반화 향상
- Phase 1은 Mixup 없이 sparse label (Head 초기화)
- Phase 2~4는 Mixup + soft label

In [ ]:
def mixup_batch(images, labels, num_classes=NUM_CLASSES, alpha=0.2):
    batch_size = len(images)
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = np.random.permutation(batch_size)
    mixed_images = lam * images + (1 - lam) * images[idx]
    labels_oh    = np.eye(num_classes)[labels.astype(int)]
    mixed_labels = lam * labels_oh + (1 - lam) * labels_oh[idx]
    return mixed_images.astype(np.float32), mixed_labels.astype(np.float32)

def make_mixup_generator(images, labels, augmentor, batch_size=32, alpha=0.2):
    gen = augmentor.flow(images, labels, batch_size=batch_size, seed=1)
    while True:
        batch_x, batch_y = next(gen)
        yield mixup_batch(batch_x, batch_y, alpha=alpha)

img_gen_base = tf.keras.preprocessing.image.ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=25,
    width_shift_range=0.15,
    height_shift_range=0.15,
    brightness_range=(0.7, 1.3),
    zoom_range=0.2,
    shear_range=10,
    fill_mode='nearest'
)

train_gen_p1 = img_gen_base.flow(train_data_f, train_labels, batch_size=32, seed=1)
train_gen_p2 = make_mixup_generator(train_data_f, train_labels, img_gen_base, alpha=0.2)

steps_per_epoch = len(train_data) // 32
print(f'steps_per_epoch: {steps_per_epoch}')
print('제너레이터 준비 완료')

## Step 5. MobileNetV3Large 모델 구성 (Phase 1: 백본 고정)
- MobileNetV3Large: ~5.4M 파라미터 (B3 12M 대비 ~55% 감소)
- TFLite 배포를 전제로 설계된 아키텍처 — Hard-swish, SE block 내장
- Dropout 0.2 (MV3는 자체적으로 효율적인 구조라 과도한 Dropout 불필요)

In [ ]:
inputs     = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
base_model = keras.applications.MobileNetV3Large(
    weights='imagenet',
    include_top=False,
    input_tensor=inputs,
    include_preprocessing=True   # raw [0,255] → 내부에서 [-1,1] 정규화
)
base_model.trainable = False

x = base_model.output
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

trainable_cnt = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'[Phase 1] 학습 가능: {trainable_cnt:,} / 전체: {model.count_params():,}')

## Step 6. Phase 1: Head 학습 (백본 고정)
- sparse label + `SparseCategoricalCrossentropy`
- `lr=1e-3`, 10 epochs

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

callbacks_p1 = [
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-6
    )
]

print('=== Phase 1: Head 학습 (백본 고정) ===')
history_p1 = model.fit(
    train_gen_p1,
    epochs=10,
    validation_data=(test_data_f, test_labels),
    callbacks=callbacks_p1
)

_, acc_p1 = model.evaluate(test_data_f, test_labels, verbose=0)
print(f'\nPhase 1 완료 - 정확도: {acc_p1:.4f}')

## Step 7. Phase 2: 전체 Fine-tuning + Mixup + Label Smoothing
- 백본 전체 Unfreeze (BatchNorm 고정 유지)
- Mixup soft label → `CategoricalCrossentropy(label_smoothing=0.1)`
- `lr=2e-5`, EarlyStopping(patience=10)

In [ ]:
for layer in base_model.layers:
    if not isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = True

trainable_cnt = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f'[Phase 2] 학습 가능: {trainable_cnt:,}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=2e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

savedModelName_p2 = 'EX16_RPS_MobileNetV3L_best.h5'
callbacks_p2 = [
    keras.callbacks.ModelCheckpoint(
        savedModelName_p2, save_best_only=True, monitor='val_accuracy', verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=5, verbose=1, min_lr=1e-8
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1
    )
]

print('\n=== Phase 2: 전체 Fine-tuning + Mixup + Label Smoothing ===')
history_p2 = model.fit(
    train_gen_p2,
    steps_per_epoch=steps_per_epoch,
    epochs=25,
    validation_data=(test_data_f, test_labels_oh),
    callbacks=callbacks_p2
)

_, acc_p2 = model.evaluate(test_data_f, test_labels_oh, verbose=0)
print(f'\nPhase 2 완료 - 정확도: {acc_p2:.4f}')

## Step 8. Phase 3: Pruning Fine-tuning (30% → 50% Sparsity)
- 가중치 50%를 0으로 제거 → 양자화와 결합 시 모델 크기 절감 극대화
- `PolynomialDecay`: 점진적으로 sparsity 증가 (급격한 정확도 저하 방지)
- `UpdatePruningStep` 콜백 필수

In [ ]:
PRUNING_EPOCHS = 15
end_step = steps_per_epoch * PRUNING_EPOCHS

pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.30,
        final_sparsity=0.50,
        begin_step=0,
        end_step=end_step
    )
}

model_pruned = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

model_pruned.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=['accuracy']
)

train_gen_p3 = make_mixup_generator(train_data_f, train_labels, img_gen_base, alpha=0.2)

callbacks_p3 = [
    tfmot.sparsity.keras.UpdatePruningStep(),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=4, verbose=1, min_lr=1e-9
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1
    )
]

print('=== Phase 3: Pruning Fine-tuning (30% → 50% Sparsity) ===')
history_p3 = model_pruned.fit(
    train_gen_p3,
    steps_per_epoch=steps_per_epoch,
    epochs=PRUNING_EPOCHS,
    validation_data=(test_data_f, test_labels_oh),
    callbacks=callbacks_p3
)

_, acc_p3 = model_pruned.evaluate(test_data_f, test_labels_oh, verbose=0)
print(f'\nPhase 3 완료 - 정확도: {acc_p3:.4f}')

## Step 9. Pruning Strip → QAT 적용
- `strip_pruning`: 불필요한 pruning wrapper 제거
- `quantize_model`: QAT wrapper 추가 → INT8 양자화 학습 준비
- QAT 적용이 실패할 경우 PTQ(representative dataset)로 자동 전환

In [ ]:
model_stripped = tfmot.sparsity.keras.strip_pruning(model_pruned)
print(f'Stripped 모델 파라미터: {model_stripped.count_params():,}')

use_qat = False
try:
    model_qat = tfmot.quantization.keras.quantize_model(model_stripped)
    model_qat.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-6),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )
    use_qat = True
    print('QAT 적용 성공 → Phase 4 QAT fine-tuning 진행')
except Exception as e:
    print(f'QAT 적용 실패: {e}')
    print('PTQ 방식으로 TFLite 변환 시 처리합니다.')
    model_qat = model_stripped
    model_qat.compile(
        optimizer=keras.optimizers.Adam(learning_rate=5e-6),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=['accuracy']
    )

print(f'use_qat = {use_qat}')

## Step 10. Phase 4: QAT Fine-tuning
- Label Smoothing 0.05 (더 정밀하게)
- `lr=5e-6`, 10 epochs
- QAT 실패 시 model_stripped을 그대로 fine-tuning (PTQ로 TFLite 변환)

In [ ]:
train_gen_p4 = make_mixup_generator(train_data_f, train_labels, img_gen_base, alpha=0.2)

callbacks_p4 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, verbose=1, min_lr=1e-9
    )
]

phase4_label = 'QAT Fine-tuning' if use_qat else 'Fine-tuning (PTQ 준비)'
print(f'=== Phase 4: {phase4_label} ===')
history_p4 = model_qat.fit(
    train_gen_p4,
    steps_per_epoch=steps_per_epoch,
    epochs=10,
    validation_data=(test_data_f, test_labels_oh),
    callbacks=callbacks_p4
)

_, acc_p4 = model_qat.evaluate(test_data_f, test_labels_oh, verbose=0)
print(f'\nPhase 4 완료 - 정확도: {acc_p4:.4f}')

## Step 11. INT8 TFLite 변환
- QAT: `converter.optimizations` 설정만으로 INT8 변환
- PTQ fallback: `representative_dataset`으로 활성화 범위 보정

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model_qat)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

if not use_qat:
    # PTQ: 활성화 범위 보정을 위한 대표 데이터셋
    def representative_data_gen():
        for i in range(min(200, len(train_data_f))):
            yield [np.expand_dims(train_data_f[i], axis=0)]
    converter.representative_dataset = representative_data_gen
    converter.inference_input_type  = tf.float32
    converter.inference_output_type = tf.float32

tflite_model = converter.convert()

model_size_mb = len(tflite_model) / 1024 / 1024
quant_type = 'QAT INT8' if use_qat else 'PTQ INT8'
print(f'TFLite ({quant_type}) 크기: {len(tflite_model):,} bytes ({model_size_mb:.2f} MB)')

save_dir = '/content/drive/MyDrive/files/save/' if colab else '../files/save/'
pathlib.Path(save_dir).mkdir(exist_ok=True, parents=True)

tfliteFileName = 'EX16_RPS_MobileNetV3L_int8.tflite'
with open(save_dir + tfliteFileName, 'wb') as f:
    f.write(tflite_model)

print(f'저장 완료: {save_dir + tfliteFileName}')

## Step 12. TFLite 정확도 평가

In [ ]:
def evaluate_tflite(model_content, test_data, test_labels):
    interpreter = tf.lite.Interpreter(model_content=model_content)
    interpreter.allocate_tensors()
    input_details  = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    correct = 0
    for image, label in zip(test_data, test_labels):
        img = np.expand_dims(image.astype(np.float32), axis=0)
        interpreter.set_tensor(input_details['index'], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        if np.argmax(output) == label:
            correct += 1
    return correct / len(test_labels)

ex16_tflite_acc = evaluate_tflite(tflite_model, test_data_f, test_labels)
print(f'EX_16 TFLite INT8 정확도: {ex16_tflite_acc:.4f}')

## Step 13. 전체 학습 곡선 시각화 (Phase 1 → 2 → 3(Pruning) → 4(QAT))

In [ ]:
acc_all      = (history_p1.history['accuracy']     + history_p2.history['accuracy']     +
                history_p3.history['accuracy']     + history_p4.history['accuracy'])
val_acc_all  = (history_p1.history['val_accuracy'] + history_p2.history['val_accuracy'] +
                history_p3.history['val_accuracy'] + history_p4.history['val_accuracy'])
loss_all     = (history_p1.history['loss']         + history_p2.history['loss']         +
                history_p3.history['loss']         + history_p4.history['loss'])
val_loss_all = (history_p1.history['val_loss']     + history_p2.history['val_loss']     +
                history_p3.history['val_loss']     + history_p4.history['val_loss'])

p1_end = len(history_p1.history['accuracy'])
p2_end = p1_end + len(history_p2.history['accuracy'])
p3_end = p2_end + len(history_p3.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('EX_16 학습 곡선 (Phase 1 → 2 → 3(Pruning) → 4(QAT))', fontsize=13)

for ax, (y_train, y_val, title) in zip(axes, [
    (acc_all, val_acc_all, 'Accuracy'),
    (loss_all, val_loss_all, 'Loss')
]):
    ax.plot(y_train, 'b', label='Train')
    ax.plot(y_val,   'g', label='Validation')
    ax.axvline(x=p1_end - 0.5, color='r',      linestyle='--', label='Phase 1→2')
    ax.axvline(x=p2_end - 0.5, color='orange', linestyle='--', label='Phase 2→3 (Pruning)')
    ax.axvline(x=p3_end - 0.5, color='purple', linestyle='--', label='Phase 3→4 (QAT)')
    ax.set_title(title); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.show()

## Step 14. 예측 시각화

In [ ]:
y_out  = model_qat.predict(test_data_f)
y_pred = np.argmax(y_out, axis=1)

size = test_data.shape[0] // 10
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('EX_16: MobileNetV3Large 예측 결과 (녹색=정답, 적색=오답)', fontsize=13)

for idx in range(10):
    cnt = idx * size + size // 2
    ax  = axes[idx // 5, idx % 5]
    ax.imshow(test_data[cnt])
    color = 'green' if y_pred[cnt] == test_labels[cnt] else 'red'
    ax.set_title(
        f'GT: {class_names[test_labels[cnt]]}\nPred: {class_names[y_pred[cnt]]}',
        color=color, fontsize=10
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

overall_acc = (y_pred == test_labels).mean()
print(f'\n전체 정확도 (float / model_qat): {overall_acc:.4f}')
print(f'전체 정확도 (TFLite INT8)       : {ex16_tflite_acc:.4f}')

## Step 15. 최종 비교
> EX_15 TFLite 크기는 EX_15 노트북 실행 후 `EX15_size_bytes` 변수에 실제 값을 입력하세요.

In [ ]:
EX15_tflite_acc   = 0.9900               # EX_15 실행 후 실제 값으로 교체
EX15_size_bytes   = int(48.0 * 1024 * 1024)  # float32 TFLite 근사값

results = [
    ('EX_12: DenseNet121 + QAT',                    0.9927, 7_703_840),
    ('EX_14: MobileNetV2 + QAT (224px)',            0.8860, int(5.12 * 1024 * 1024)),
    ('EX_15: EfficientNetB3 float32',               EX15_tflite_acc, EX15_size_bytes),
    ('EX_16: MobileNetV3Large + Pruning + INT8',    ex16_tflite_acc, len(tflite_model)),
]

print('=' * 72)
print(f'{"모델":<50} {"정확도":>8} {"크기(MB)":>10}')
print('-' * 72)
for name, acc, sz in results:
    print(f'{name:<50} {acc:>8.4f} {sz / 1024 / 1024:>10.2f}')
print('=' * 72)

print('\n[EX_16 적용 기법 요약]')
print('  1. MobileNetV3Large 백본 (5.4M params, TFLite 최적화 설계)')
print('  2. IMG_SIZE=224 (MV3 기본, B3 300 대비 연산량 절감)')
print('  3. Mixup 증강 (soft label, alpha=0.2)')
print('  4. Label Smoothing 0.1 → 0.05 (단계적 정밀화)')
print('  5. 4단계 Fine-tuning: Head → 전체 → Pruning → QAT')
print('  6. 가중치 Pruning 50% Sparsity (PolynomialDecay)')
print(f'  7. {"QAT" if use_qat else "PTQ"} INT8 TFLite (float32 대비 ~4x 크기 절감)')